# Stage 1 — Predict Decline

**What this notebook does:** estimates how much each of the 5 cognitive domains is likely to **worsen over the next year** for a patient. That "will-worsen" signal becomes **one input to the SAW recommender** in Stage 2 (which combines *weak now* + *will worsen* to choose what to train).

**Why a separate notebook:** Stage 0 made *one row per person* (a snapshot). Decline is **change between visits**, so here we use the **longitudinal** data — every yearly visit — and reuse the saved healthy ruler (`models/zscore_reference.json`) so scores are on the exact same scale.

**Plan:** 1) build the decline dataset (score every visit) → 2) build the decline signal → 3) validate → 4) save for SAW.

## Setup

Load the tools, the file path, the 5-domain definitions, the valid score ranges, and the **saved z-ruler** from Stage 0. Loading the same ruler is the "same-scale contract": a patient's scores must be measured against the identical healthy baseline the project trained on.

In [ ]:
import os, json
import numpy as np, pandas as pd

PATH = r"D:\SLIIT\Final Research\NeuroSage-cognitive\cognitive-train-system\data\investigator_ftldlbd_nacc74.csv"
PROJ = os.path.dirname(os.path.dirname(PATH))            # .../cognitive-train-system
REF_PATH = os.path.join(PROJ, "models", "zscore_reference.json")

ID, VISIT, VER, TIME = "NACCID", "NACCVNUM", "FORMVER", "NACCFDYS"   # NACCFDYS = days since first visit

NEURO = {
    "memory":       ["CRAFTVRS", "CRAFTURS", "CRAFTDVR", "CRAFTDRE", "UDSBENTD"],
    "attention":    ["DIGFORCT", "DIGFORSL", "DIGBACCT", "DIGBACLS", "TRAILA"],
    "executive":    ["TRAILB", "UDSVERFC", "UDSVERLC", "UDSVERTN"],
    "language":     ["MINTTOTS", "ANIMALS", "VEG"],
    "visuospatial": ["UDSBENTC"],
}
NEURO_COLS = sum(NEURO.values(), [])

RANGES = {
    "CRAFTVRS": (0, 44), "CRAFTURS": (0, 25), "CRAFTDVR": (0, 44), "CRAFTDRE": (0, 25),
    "UDSBENTD": (0, 17), "UDSBENTC": (0, 17),
    "DIGFORCT": (0, 14), "DIGFORSL": (0, 9), "DIGBACCT": (0, 14), "DIGBACLS": (0, 8),
    "TRAILA": (0, 150), "TRAILB": (0, 300),
    "UDSVERFC": (0, 40), "UDSVERLC": (0, 40), "UDSVERTN": (0, 80),
    "MINTTOTS": (0, 32), "ANIMALS": (0, 77), "VEG": (0, 77),
}

reference = json.load(open(REF_PATH))
print("Loaded z-ruler for", len(reference), "tests")
print("From:", REF_PATH)

## Step 1 — Build the decline dataset (score every visit)

To see decline we need each person's **whole history**, not one snapshot. This cell:

1. Loads the columns we need for **all** visits.
2. Keeps UDS **v3 + v4** and recodes "not done" codes to blanks (same rules as Stage 0).
3. Rebuilds the **same two-arm cohort** (AD / LBD, drop mixed-AD) — the counts should match Stage 0 (AD 9,882 / LBD 1,084).
4. Converts every visit's test scores to **z-scores** with the saved ruler (flipping the timed Trails), then averages them into the **5 domain scores per visit**.
5. Saves a tidy **longitudinal table** — one row per visit: `NACCID, visit, days, age, 5 domains, diagnosis` — to `data/` (gitignored).

The result (`long`) is the "photo album" the decline signal will learn from. We also report how many people have **≥ 2 scored visits**, since you need at least two points to measure change.

In [ ]:
# 1) load the columns we need, for ALL visits (longitudinal)
cols = [ID, VISIT, VER, TIME, "NACCAGE", "NACCETPR", "CVDIF"] + NEURO_COLS
raw = pd.read_csv(PATH, usecols=cols, low_memory=False)
for c in raw.columns:
    if c != ID:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

# 2) keep UDS v3+v4, recode "not done" codes -> NaN
v = raw[VER].astype(str).str.strip()
uds = raw[v.str.startswith("3") | v.str.startswith("4")].copy()
for col in NEURO_COLS:
    lo, hi = RANGES[col]
    uds[col] = uds[col].where(uds[col].between(lo, hi))

# 3) two-arm cohort: most-recent etiology; drop mixed-AD, keep mixed-LBD
last = uds.sort_values(VISIT).groupby(ID)["NACCETPR"].last()
AD_ids, LBD_ids = last.index[last == 1], last.index[last == 2]
vasc_ever = uds.groupby(ID)["CVDIF"].apply(lambda s: s.isin([1, 2]).any())
AD_clean  = AD_ids[~vasc_ever.reindex(AD_ids).fillna(False).values]
keep_ids  = AD_clean.union(LBD_ids)
diagnosis = pd.Series(index=keep_ids, dtype=object)
diagnosis.loc[AD_clean] = "AD"
diagnosis.loc[LBD_ids]  = "LBD"

# 4) score EVERY cohort visit with the saved ruler (flip the timed Trails)
coh = uds[uds[ID].isin(keep_ids)].copy()
for col, r in reference.items():
    if col in coh.columns and r["sd"] > 0:
        z = (coh[col] - r["mean"]) / r["sd"]
        coh[col] = -z if not r["higher_is_better"] else z

# 5) one row per visit: id, visit, days-since-first-visit, age, 5 domain z-scores, diagnosis
long = pd.DataFrame({ID: coh[ID].values, "visit": coh[VISIT].values,
                     "days": coh[TIME].values, "age": coh["NACCAGE"].values})
for d, cs in NEURO.items():
    use = [c for c in cs if c in coh.columns]
    long[d] = coh[use].mean(axis=1, skipna=True).values
long["diagnosis"] = long[ID].map(diagnosis)

OUT = os.path.join(PROJ, "data", "longitudinal_scores.parquet")
long.to_parquet(OUT, index=False)
print("Saved:", OUT)

# summary
dom = list(NEURO)
vpp = long.dropna(subset=dom, how="all").groupby(ID).size()   # scored visits per person
print(f"\nCohort: {long[ID].nunique():,} people | {len(long):,} visit-rows")
print(f"Diagnosis (people): {diagnosis.value_counts().to_dict()}")
print(f"Scored visits per person: min {vpp.min()}, median {int(vpp.median())}, max {vpp.max()}")
print(f"People with >=2 scored visits (usable for decline): {(vpp >= 2).sum():,}")
print("\nPreview (one person's history):")
print(long.sort_values([ID, 'visit']).head(8).round(2).to_string(index=False))

## Step 2 — How much decline data do we have per disease?

Before choosing *how* to build the decline signal, we check **how many people we can learn from in each disease group**. A trained model needs enough examples; if the LBD group is tiny, a model for it would be unreliable and we would prefer simple averaged decline rates instead.

Among the people with ≥ 2 scored visits, this cell reports:
- how many are **AD** vs **LBD**,
- how many **visit-to-visit intervals** each arm gives (roughly how many change-measurements a model could train on),
- the typical **follow-up length** in years.

In [ ]:
dom = list(NEURO)
scored = long.dropna(subset=dom, how="all")          # visits that have >=1 domain score
vpp    = scored.groupby(ID).size()                   # scored visits per person
usable = vpp.index[vpp >= 2]                          # need >=2 to measure change

dx  = long.drop_duplicates(ID).set_index(ID)["diagnosis"]
udx = dx.reindex(usable)

print(f"Usable for decline (>=2 scored visits): {len(usable):,}")
print(f"   by arm: {udx.value_counts().to_dict()}")

pairs = vpp.reindex(usable) - 1                       # k visits -> k-1 intervals
print(f"   visit-to-visit intervals per arm (approx training points): "
      f"{pairs.groupby(udx).sum().astype(int).to_dict()}")

span = scored[scored[ID].isin(usable)].groupby(ID)["days"].agg(lambda s: (s.max() - s.min()) / 365.25)
print(f"   median follow-up (years) per arm: "
      f"{span.groupby(dx.reindex(span.index)).median().round(1).to_dict()}")

## Step 3 — Build the target (decline slopes) and the feature table

Now we turn each person's visit history into **one row a model can learn from**.

**Target (what we predict) — annual decline per domain.** For each person and domain we fit a straight line through their yearly z-scores and take its **slope** (z-score change per year); negative = getting worse. A person needs ≥ 2 scored visits for that domain, or the slope is left blank — we never invent an answer.

**Features (what we predict *from*) — the starting point.** From each person's **first** visit we take the 5 baseline domain scores, the diagnosis (AD/LBD) and age; plus sex and education (from the Stage 0 profile table).

**Preprocessing done here:** we first **clip** domain z-scores to −4…+4 so ceiling-effect outliers (like the −5.65 language score) can't distort the slopes.

We save `data/decline_model_table.parquet` and run a **sanity check**: average decline by arm should be mostly negative, with AD worst in memory and LBD worst in visuospatial — the Stage 0 signatures, now seen *over time*.

In [ ]:
dom = list(NEURO)

# --- 3a. TARGET: annual decline slope (z per year) per person, per domain ---
L = long.sort_values([ID, "visit"]).copy()
L[dom] = L[dom].clip(-4, 4)                         # tame ceiling-effect outliers

def annual_slope(days, vals):
    m = vals.notna()
    if m.sum() < 2:
        return np.nan
    yrs = (days[m] - days[m].min()) / 365.25
    if yrs.max() < 0.25:                            # need a real time gap
        return np.nan
    return np.polyfit(yrs.values, vals[m].values, 1)[0]   # slope of best-fit line

slopes = (L.groupby(ID)
            .apply(lambda g: pd.Series({d: annual_slope(g["days"], g[d]) for d in dom}))
            .add_prefix("decline_"))

# --- 3b. FEATURES: each person's FIRST (baseline) visit + demographics ---
base = L.groupby(ID).head(1).set_index(ID)          # earliest visit row per person
prof = pd.read_parquet(os.path.join(PROJ, "data", "cognitive_profiles.parquet")).set_index("NACCID")

feat = pd.DataFrame(index=base.index)
for d in dom:
    feat["base_" + d] = base[d]
feat["diagnosis"] = base["diagnosis"]
feat["age"]       = base["age"]
feat = feat.join(prof[["sex", "education"]])

# --- combine into one modelling table; keep people who have >=1 target ---
tgt_cols = ["decline_" + d for d in dom]
model_df = feat.join(slopes)
model_df = model_df[model_df[tgt_cols].notna().any(axis=1)]

OUT = os.path.join(PROJ, "data", "decline_model_table.parquet")
model_df.reset_index().to_parquet(OUT, index=False)
print("Saved:", OUT)

print(f"\nModelling table: {len(model_df):,} people x {model_df.shape[1]} columns")
print("People with a target, per domain, by arm:")
for d in dom:
    c = model_df.dropna(subset=["decline_" + d]).groupby("diagnosis").size().to_dict()
    print(f"   {d:<13} {c}")

print("\nSanity - mean annual decline (z/year) by arm (negative = worsening):")
print(model_df.groupby("diagnosis")[tgt_cols].mean().round(3).to_string())

## Step 4 — Modelling: preprocessing & feature prep

We predict each domain's decline **separately** (5 small regression tasks), and **separately for AD and LBD** so the large AD group can't drown out LBD.

This cell prepares the inputs:
- **encode** sex as 0/1 (`sex01`),
- list the **8 features**: the 5 baseline domain scores + age + education + sex,
- show how much is **missing**.

Missing values are **not** filled here, on purpose. Imputation happens **inside cross-validation** (next cell), learning the fill values from training data only. Filling them now, using the whole dataset, would leak test information and make our scores look better than they really are.

In [ ]:
from sklearn import __version__ as skv
print("scikit-learn", skv)

dom = list(NEURO)
mdf = model_df.copy()
mdf["sex01"] = mdf["sex"].map({"M": 1, "F": 0})
feat_cols = ["base_" + d for d in dom] + ["age", "education", "sex01"]

print("\nFeatures (8):", feat_cols)
print("\nMissing values in features (%):")
print((mdf[feat_cols].isna().mean() * 100).round(1).to_string())

## Step 4 (cont.) — Split, train, evaluate (baseline vs Ridge)

We use **5-fold cross-validation**: the people are split into 5 groups; each takes a turn as the test set while the model trains on the other four. This uses our limited data fairly and gives a stable error estimate — important for the small LBD group. (Each person is one row, so splitting rows = splitting people → no leakage.)

For every arm × domain we compare two predictors:
- **Baseline** — always predict that group's *average* decline (the bar to beat).
- **Ridge regression** — a regularised linear model, with imputation + scaling done **inside each fold** (leak-free).

We report **MAE** (mean absolute error, z/year — lower is better) for both and the **% improvement** of Ridge over the baseline. Honest expectation: individual decline is noisy, so gains may be small — and a small or zero gain is itself a valid result (it would tell us the transparent average-rate approach is good enough).

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error

def ridge():
    return make_pipeline(SimpleImputer(strategy="median"),
                         StandardScaler(),
                         RidgeCV(alphas=np.logspace(-2, 3, 20)))

rows = []
for arm in ["AD", "LBD"]:
    sub = mdf[mdf["diagnosis"] == arm]
    for d in dom:
        t = "decline_" + d
        data = sub[sub[t].notna()]
        X, y = data[feat_cols], data[t].values
        cv = KFold(5, shuffle=True, random_state=0)
        base = cross_val_predict(DummyRegressor(strategy="mean"), X, y, cv=cv)
        pred = cross_val_predict(ridge(), X, y, cv=cv)
        b, m = mean_absolute_error(y, base), mean_absolute_error(y, pred)
        rows.append({"arm": arm, "domain": d, "n": len(y),
                     "baseline_MAE": round(b, 3), "ridge_MAE": round(m, 3),
                     "improve_%": round(100 * (b - m) / b, 1)})

res = pd.DataFrame(rows)
print(res.to_string(index=False))
print(f"\nMean improvement of Ridge over baseline:  "
      f"AD {res[res.arm=='AD']['improve_%'].mean():.1f}%   "
      f"LBD {res[res.arm=='LBD']['improve_%'].mean():.1f}%")

## Step 4c — Verification (1): is the improvement real, or luck?

Beating the baseline in all 10 comparisons is encouraging, but it does not yet prove the gain is dependable. We check two things:

1. **Consistency across folds** — in how many of the 5 folds did Ridge actually win? 5/5 is far more convincing than 3/5.
2. **A confidence interval** — we resample the patients 2,000 times (a *bootstrap*) and look at the spread of Ridge's error reduction. If the 95% interval stays **above zero**, the improvement is reliable, not a fluke. Each patient is one row, so this resampling respects the design.

How to read the table: `MAE_gain` = how much lower Ridge's error is (z/year, higher = better); `reliable = yes` means the whole 95% interval sits above zero.

In [ ]:
from sklearn.model_selection import KFold

def eval_task(X, y):
    """5-fold CV returning out-of-fold predictions for baseline + Ridge, and per-fold MAE gain."""
    cv = KFold(5, shuffle=True, random_state=0)
    bp = np.zeros(len(y)); rp = np.zeros(len(y)); fold_gain = []
    for tr, te in cv.split(X):
        bm = DummyRegressor(strategy="mean").fit(X.iloc[tr], y[tr])
        rm = ridge().fit(X.iloc[tr], y[tr])
        pb, pr = bm.predict(X.iloc[te]), rm.predict(X.iloc[te])
        bp[te], rp[te] = pb, pr
        fold_gain.append(mean_absolute_error(y[te], pb) - mean_absolute_error(y[te], pr))
    return bp, rp, np.array(fold_gain)

rng = np.random.default_rng(0)
rows = []
for arm in ["AD", "LBD"]:
    sub = mdf[mdf["diagnosis"] == arm]
    for d in dom:
        data = sub[sub["decline_" + d].notna()]
        X, y = data[feat_cols], data["decline_" + d].values
        bp, rp, fgain = eval_task(X, y)
        diff = np.abs(y - bp) - np.abs(y - rp)                 # per-person error reduction
        boot = np.array([diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(2000)])
        lo, hi = np.percentile(boot, [2.5, 97.5])
        rows.append({"arm": arm, "domain": d, "n": len(y),
                     "MAE_gain": round(diff.mean(), 4),
                     "CI_low": round(lo, 4), "CI_high": round(hi, 4),
                     "folds_won": f"{int((fgain > 0).sum())}/5",
                     "reliable": "yes" if lo > 0 else "no"})
print(pd.DataFrame(rows).to_string(index=False))

## Step 4c — Verification (2): does the model rank domains correctly?

This is the test that matters most, because SAW does not use the exact decline number — it uses the **order** of which domains will worsen fastest. A lower error does not automatically mean a better order.

For every patient who has all 5 domains, we compare:
- the **order the model predicts** against the **order that actually happened** (a rank correlation, −1…+1, higher = better),
- and, as a benchmark, the **fixed disease-average order** (the same ranking for everyone in that arm).

If the model's ranking beats the fixed group ranking, personalising per patient genuinely helps the recommender. If they are about equal, the honest conclusion is that the group-average order is good enough, and SAW could use that simpler signal.

In [ ]:
from scipy.stats import spearmanr

for arm in ["AD", "LBD"]:
    sub = mdf[mdf["diagnosis"] == arm]
    preds, obs = {}, {}
    for d in dom:
        data = sub[sub["decline_" + d].notna()]
        X, y = data[feat_cols], data["decline_" + d].values
        _, rp, _ = eval_task(X, y)
        preds[d] = pd.Series(rp, index=data.index)
        obs[d]   = pd.Series(y,  index=data.index)
    P, O = pd.DataFrame(preds), pd.DataFrame(obs)
    keep = O.dropna().index.intersection(P.dropna().index)     # patients with all 5 domains
    P, O = P.loc[keep], O.loc[keep]
    gmean = O.mean().values                                     # fixed disease-average order

    rho_model = np.array([spearmanr(P.loc[i].values, O.loc[i].values).correlation for i in keep])
    rho_group = np.array([spearmanr(gmean,            O.loc[i].values).correlation for i in keep])

    print(f"{arm}: {len(keep)} patients with all 5 domains")
    print(f"   rank-corr  model     vs observed : {np.nanmean(rho_model): .3f}")
    print(f"   rank-corr  group-avg vs observed : {np.nanmean(rho_group): .3f}")
    print(f"   model ranks better than group-avg for {np.mean(rho_model > rho_group) * 100:.0f}% of patients\n")